# Importations and definitions

In [1]:
# Numerical and scientific python programming
import numpy as np
from scipy.optimize import minimize

# Auxiliary python functions
from typing import Tuple, Dict, Callable, Any
from dataclasses import dataclass

# Local importations
from moments.bloch import compute_pauli_basis, compute_bloch_vector, compute_tensor_basis, compute_subset_index_map, compute_bloch_norms_from_vector
from moments.initialization import compute_initial_param, compute_X_from_param, compute_dm_from_X
from moments.quantum import compute_is_valid_dm, compute_concurrence, compute_eof

In [2]:
@dataclass
class OptimizationResult:
    rho_initial: np.ndarray
    rho_final: np.ndarray
    bloch_initial: Dict[Tuple[int, ...], np.ndarray]
    bloch_final: Dict[Tuple[int, ...], np.ndarray]
    moments_initial: Dict[Tuple[int, ...], float]
    moments_final: Dict[Tuple[int, ...], float]
    metric_name: str
    metric_initial: float
    metric_final: float
    checks: Dict[str, Any]
    optimizer_info: Dict[str, Any]

# Utility functions for optimization

In [3]:
def choose_metric(metric: str) -> Callable:
    if metric == "concurrence":
        return compute_concurrence
    elif metric in {"eof", "entanglement_of_formation", "entanglement-of-formation"}:
        return compute_eof
    else:
        raise ValueError("Metric must be 'concurrence' or 'eof'.")

def trivial_result(tensor_basis: np.ndarray, subset_index_map: Dict[Tuple[int, ...], np.ndarray], rho: np.ndarray,
                   Rt: Dict[Tuple[int, ...], float], metric: str, reason: str) -> OptimizationResult:
    
    compute_metric = choose_metic(metric)
    r0 = compute_bloch_vector(tensor_basis, subset_index_map, rho)
    R0 = compute_bloch_norms_from_vector(r0)
    metric_value = compute_metric(metric)

    moments_difference = np.mean([R0[subset] - Rt[subset] for subset in subset_index_map.keys()])

    return OptimizationResult(
        rho_initial=rho,
        rho_final=rho,
        bloch_initial=r0,
        bloch_final=r0,
        moments_initial=R0,
        moments_final=R0,
        metric_name=metric,
        metric_initial=metric_value,
        metric_final=metric_value,
        checks={"moments_difference": moments_difference, "is_valid_dm": True, "note": reason,},
        optimizer_info={"mode": "trivial", "reason": reason,},)

# Main optimizer

In [4]:
def optimize_moment_preserving_entanglement(d: int, tensor_basis: np.ndarray, subset_index_map: Dict[Tuple[int, ...], np.ndarray], Rt: Dict[Tuple[int, ...], float],
    metric: str = "concurrence", purity_tol: float = 1e-10, psd_tol: float = 1e-10, local_maxiter: int = 500) -> OptimizationResult:
    
    # --- 1. Find valid initial state ---
    x0 = None
    rho0 = None
    initial_state = False
    while not initial_state:
        x0 = compute_initial_param(d, tensor_basis, subset_index_map, Rt)
        rho0 = compute_dm_from_X(compute_X_from_param(x0))
        initial_state = compute_is_valid_dm(rho0, psd_tol)
    
    if x0 is None or rho0 is None:
        raise RuntimeError("Failed to compute initial state")

    r0 = compute_bloch_vector(tensor_basis, subset_index_map, rho0)
    R0 = compute_bloch_norms_from_vector(r0)
    
    # --- 2. Check trivial cases ---
    SR = sum([Rt[subset]**2 for subset in subset_index_map.keys()])
    
    if SR >= d - 1 - purity_tol:
        return trivial_result(
            tensor_basis, subset_index_map, rho0, Rt, metric,
            reason="The input state is numerically pure. Entanglement cannot be improved."
        )

    if SR <= 0 + purity_tol:
        return trivial_result(
            tensor_basis, subset_index_map, np.identity(d), Rt, metric,
            reason="The input state is numerically maximally mixed. Entanglement cannot be improved."
        )
        
    # --- 3. Shared cache for efficiency ---
    cache = {}
    def compute_shared(x):
        if "x" not in cache or not np.allclose(x, cache["x"]):
            rho = compute_dm_from_X(compute_X_from_param(x))
            r = compute_bloch_vector(tensor_basis, subset_index_map, rho)
            R = compute_bloch_norms_from_vector(r)
            
            cache["x"] = x.copy()
            cache["rho"] = rho.copy()
            cache["R"] = R.copy()
        
        return cache["rho"], cache["R"]
    
    # --- 4. Objective function ---
    compute_metric = choose_metric(metric)
    
    def objective(x):
        rho, _ = compute_shared(x)
        return -compute_metric(rho)
    
    metric0 = compute_metric(rho0)
    
    # --- 5. Constraints (one per subsystem) ---
    constraints = []
    
    for subset in subset_index_map.keys():
        target = Rt[subset]
        
        def make_constraint(subset, target):
            def constraint(x):
                _, R = compute_shared(x)
                return R[subset] - target
            return constraint
        
        constraints.append({
            "type": "eq",
            "fun": make_constraint(subset, target)
        })
    
    # --- 6. Run optimization ---
    result = minimize(objective, x0,
        method="SLSQP", constraints=constraints,
        options={
            "maxiter": local_maxiter,
            "ftol": 1e-12,
            "disp": False
        }
    )
    
    # --- 7. Extract final state ---
    rho_opt, R_opt = compute_shared(result.x)
    r_opt = compute_bloch_vector(tensor_basis, subset_index_map, rho_opt)
    metric_opt = compute_metric(rho_opt)

    # --- 8. Run checks on the solution ---
    R_match = {subset: float(abs(R_opt[subset] - Rt[subset]))
               for subset in subset_index_map.keys()}
    moments_equal = (np.array(list(R_match.values())) <= 1e-8).all()
    is_valid_dm, is_valid_dm_info = compute_is_valid_dm(rho_opt, psd_tol, True)
    
    
    checks = {
        "R_match": R_match,
        "moments_equal": moments_equal,
        "is_valid_dm": is_valid_dm,
        "is_valid_dm_info": is_valid_dm_info,
    }

    optimizer_info = {
        "mode": "moment_preserving_bloch",
        "metric_optimized": metric,
        "result_success": bool(result.success),
        "result_message": str(result.message),
        "fun": complex(result.fun),
        "nit": int(result.nit),
        "nfev": int(result.nfev),
    }

    return OptimizationResult(
        rho_initial=rho0,
        rho_final=rho_opt,
        bloch_initial=r0,
        bloch_final=r_opt,
        moments_initial=R0,
        moments_final=R_opt,
        metric_name=metric,
        metric_initial=metric0,
        metric_final=metric_opt,
        checks=checks,
        optimizer_info=optimizer_info,
    )

# Tests

In [5]:
N = 2
d = 2 ** N

pauli_basis = compute_pauli_basis()
local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

Rt = {(1,): 0.6, (2,): 0.6, (1, 2): 1.2}

In [6]:
optimization_result = optimize_moment_preserving_entanglement(d, tensor_basis, subset_index_map, Rt, metric="concurrence", purity_tol=1e-10, psd_tol=1e-10, local_maxiter=500)

In [7]:
checks = optimization_result.checks
print(checks["R_match"])
print(checks["is_valid_dm"])

{(1,): 1.0255559087513078e-06, (2,): 6.319868939286266e-08, (1, 2): 1.868019137152288e-06}
True


In [8]:
print("Matrix_difference:", np.allclose(optimization_result.rho_initial, optimization_result.rho_final))
print("bloch_initial:", optimization_result.bloch_initial)
print("bloch_final:", optimization_result.bloch_final)
print("moments_initial:", optimization_result.moments_initial)
print("moments_final:", optimization_result.moments_final)
print("metric_name:", optimization_result.metric_name)
print("metric_initial:", optimization_result.metric_initial)
print("metric_final:", optimization_result.metric_final)

Matrix_difference: True
bloch_initial: {(1,): array([-0.22690486,  0.25683891, -0.49249034]), (2,): array([-0.17135481, -0.25342487,  0.51615239]), (1, 2): array([ 0.72353343,  0.05844368, -0.17726179,  0.08149466,  0.0543677 ,
        0.62815435,  0.05749381,  0.61871782, -0.30226659])}
bloch_final: {(1,): array([-0.22690486,  0.25683891, -0.49249034]), (2,): array([-0.17135481, -0.25342487,  0.51615239]), (1, 2): array([ 0.72353343,  0.05844368, -0.17726179,  0.08149466,  0.0543677 ,
        0.62815435,  0.05749381,  0.61871782, -0.30226659])}
moments_initial: {(1,): np.float64(0.5999989744440912), (2,): np.float64(0.5999999368013106), (1, 2): np.float64(1.1999981319808628)}
moments_final: {(1,): np.float64(0.5999989744440912), (2,): np.float64(0.5999999368013106), (1, 2): np.float64(1.1999981319808628)}
metric_name: concurrence
metric_initial: 0.5682598657485591
metric_final: 0.5682598657485591


In [9]:
optimizer_info = optimization_result.optimizer_info
print(optimizer_info)

{'mode': 'moment_preserving_bloch', 'metric_optimized': 'concurrence', 'result_success': False, 'result_message': 'Singular matrix C in LSQ subproblem', 'fun': (-0.5682598657485591+0j), 'nit': 1, 'nfev': 33}
